# STPR-CLIP v1 — Masked Token-Level Reconstruction

**Changes:** image/text reconstruction queries get randomly masked at 15% of valid positions; add local cosine reconstruction loss (β=0.05) while preserving original global reconstruction loss and CLIP encoders. Correct AMP accumulation over four batches. Baseline and improved versions **must** use the same corrected training loop for a fair comparison.

Run cells sequentially. The notebook expects `/content/drive/MyDrive/PR-CLIP.zip` and the dataset archive already used in your previous notebook. It fetches `RET-2.pt` from Hugging Face (same URL as your supplied notebook). It does not resume from a prior checkpoint.

This is a proposed architectural modification, **not an already validated accuracy improvement**.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# upload PR-CLIP.zip lên Drive trước, sửa đường dẫn cho đúng
!rm -rf /content/PR-CLIP
!unzip -q -o /content/drive/MyDrive/PR-CLIP.zip -d /content
%cd /content/PR-CLIP

Mounted at /content/drive
/content/PR-CLIP


In [2]:
!pip install -q open_clip_torch==2.20.0 clip-benchmark==1.6.1 transformers==4.45.2 adapters timm sentence-transformers wandb pyyaml gdown
!pip install -q "protobuf>=6.31.1,<7"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
proto-plus 1.29.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-api-core 2.30.3 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-mon

In [3]:
!gdown --fuzzy "https://drive.google.com/file/d/15htMe2iE4kVujjWURxfCxZXk8vAAetoT/view" -O images.zip
!rm -rf images && mkdir images
!tar -xzf images.zip -C images
!find images -type f -exec mv -t images {} + 2>/dev/null
!find images -mindepth 1 -type d -empty -delete
!rm images.zip
!mkdir -p cache/weights

Downloading...
From (original): https://drive.google.com/uc?id=15htMe2iE4kVujjWURxfCxZXk8vAAetoT
From (redirected): https://drive.google.com/uc?id=15htMe2iE4kVujjWURxfCxZXk8vAAetoT&confirm=t&uuid=9c5ab5e7-79a5-44e8-9ad7-80eea67e33b9
To: /content/PR-CLIP/images.zip
100% 1.46G/1.46G [00:27<00:00, 52.6MB/s]


In [4]:
!mkdir -p /content/PR-CLIP/cache/weights
!wget -O /content/PR-CLIP/cache/weights/RET-2.pt \
  https://huggingface.co/Zilun/GeoRSCLIP/resolve/main/ckpt/RS5M_ViT-B-32_RET-2.pt
!ls -la /content/PR-CLIP/cache/weights/RET-2.pt   # khoảng 605MB

--2026-11-03 13:26:37--  https://huggingface.co/Zilun/GeoRSCLIP/resolve/main/ckpt/RS5M_ViT-B-32_RET-2.pt
Resolving huggingface.co (huggingface.co)... 3.171.171.104, 3.171.171.6, 3.171.171.65, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.104|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/655fb2aadc7511e22748819d/48ec26e685f2b7622075158c0649039d64cdcbdbe150faedb6e79ea1d13d0e78?xip=Sia6o1sUSEc&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27RS5M_ViT-B-32_RET-2.pt%3B+filename%3D%22RS5M_ViT-B-32_RET-2.pt%22%3B&X-Xet-Cas-Uid=public&Expires=1793715997&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU1ZmIyYWFkYzc1MTFlMjI3NDg4MTlkLzQ4ZWMyNmU2ODVmMmI3NjIyMDc1MTU4YzA2NDkwMzlkNjRjZGNiZGJlMTUwZmFlZGI2ZTc5ZWExZDEzZDBlNzhcXD94aXA9U2lhNm8xc1VTRWMmdXNlcl9pZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qJlgtWGV0LUNhcy1VaWQ9cHVibGljIiwiQ29

In [5]:
import pandas as pd, os
for f in ['dataset/train/ret2.csv','dataset/test/rsicd_test.csv','dataset/test/rsitmd_test.csv']:
    names = pd.read_csv(f)['filename'].unique()
    missing = [n for n in names if not os.path.exists(os.path.join('images', n))]
    print(f, '| tổng:', len(names), '| thiếu:', len(missing))

dataset/train/ret2.csv | tổng: 13092 | thiếu: 0
dataset/test/rsicd_test.csv | tổng: 1093 | thiếu: 0
dataset/test/rsitmd_test.csv | tổng: 452 | thiếu: 0


In [6]:
%cd /content/PR-CLIP
!sed -i 's/from timm.optim.nadam import Nadam/from timm.optim.nadam import NAdamLegacy as Nadam/' training/optimization.py
!sed -i 's/from timm.optim.radam import RAdam$/from timm.optim.radam import RAdamLegacy as RAdam/' training/optimization.py
!sed -n 5,12p training/optimization.py
!nvidia-smi | head -12
!sed -i 's/np\.string_/np.bytes_/g' data/train_data.py utils/tsne_checkpoints.py
!sed -i 's/astype(np\.float)/astype(float)/g' evaluation/linear_eval.py
!grep -n "np.bytes_" data/train_data.py

/content/PR-CLIP
from timm.optim.adafactor import Adafactor
from timm.optim.adahessian import Adahessian
from timm.optim.adamp import AdamP
from timm.optim.lookahead import Lookahead
from timm.optim.nadam import NAdamLegacy as Nadam
from timm.optim.radam import RAdamLegacy as RAdam
from timm.optim.rmsprop_tf import RMSpropTF
from timm.optim.sgdp import SGDP
Tue Nov  3 13:26:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+=============

In [7]:
!pip install -q cn_clip --no-deps
!nvidia-smi
!pip install -q faiss-cpu click

Tue Nov  3 13:26:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Repository compatibility fixes from your original notebook

In [8]:
%cd /content/PR-CLIP
import re

def read(p): return open(p, encoding='utf-8').read().replace('\r\n', '\n')
def write(p, s): open(p, 'w', encoding='utf-8').write(s)
def sub(path, pairs):
    s = read(path)
    for old, new in pairs:
        s = s.replace(old, new)
    write(path, s)


# 2. timm / numpy bản mới đổi tên + tắt log dài
sub('training/optimization.py', [
    ('from timm.optim.nadam import Nadam\n', 'from timm.optim.nadam import NAdamLegacy as Nadam\n'),
    ('from timm.optim.radam import RAdam\n', 'from timm.optim.radam import RAdamLegacy as RAdam\n'),
    ('logging.info("Param groups = %s" % json.dumps(parameter_group_names, indent=2))', 'pass'),
])
sub('data/train_data.py', [('np.string_', 'np.bytes_')])
sub('utils/tsne_checkpoints.py', [('np.string_', 'np.bytes_')])
sub('evaluation/linear_eval.py', [('astype(np.float)', 'astype(float)')])
sub('training/main.py', [
    ("logging.info(f'\\t{n}\\t{list(p.size())}')", 'pass'),
    ("logging.info('Model structure\\n' +str(model))", 'pass'),
    ('logging.info(f"  {name}: {val}")', 'pass'),
])

# 3. Thay return_hidden của open_clip bản sửa
helper = '''    def _vit_hidden(self, images):
        v = self.image_backbone
        x = v.conv1(images.type(v.conv1.weight.dtype))
        x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
        cls = v.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)
        x = torch.cat([cls, x], dim=1)
        x = x + v.positional_embedding.to(x.dtype)
        x = v.patch_dropout(x)
        x = v.ln_pre(x)
        x = x.permute(1, 0, 2)
        x = v.transformer(x)
        x = x.permute(1, 0, 2)
        return v.ln_post(x)

'''
s = read('model/model.py')
if '_vit_hidden' not in s:
    s = s.replace("self.image_backbone(images, return_hidden=True)", "self._vit_hidden(images)")
    marker = "    def forward(self, images, texts, text_only=False):"
    s = s.replace(marker, helper + marker)
    write('model/model.py', s)

# 4. In log theo % epoch (10 -> 10%,20%,...; 50 -> 50%,100%)
LOG_PCT = 10
s = read('training/train.py')
s = s.replace("(i % 10 == 0 or batch_count == num_batches_per_epoch)",
              "(batch_count in {round(num_batches_per_epoch * k * LOG_PCT / 100) for k in range(1, 100 // LOG_PCT + 1)})")
if 'LOG_PCT = ' not in s:
    anchor = "import torch.nn.functional as F\n"
    s = s.replace(anchor, anchor + f"\nLOG_PCT = {LOG_PCT}\n", 1)
else:
    s = re.sub(r"LOG_PCT = \d+", f"LOG_PCT = {LOG_PCT}", s, count=1)
write('training/train.py', s)


# Kiểm tra: phải ra 2, 1, 2, 2, True
!grep -c "_vit_hidden" model/model.py
!grep -c "NAdamLegacy" training/optimization.py
!grep -c "np.bytes_" data/train_data.py
!grep -c "LOG_PCT" training/train.py
print('[DBG-fp32]' in read('training/train.py'))


%cd /content/PR-CLIP
import re, py_compile
p = 'training/main.py'
s = open(p, encoding='utf-8').read().replace('\r\n', '\n')

if '[LITE-CKPT]' in s:
    print('Đã vá từ trước, không làm gì thêm.')
else:
    def sub_regex(pattern, repl, text, name):
        n = len(re.findall(pattern, text))
        assert n == 1, f'[{name}] khớp {n} chỗ (cần đúng 1)'
        return re.sub(pattern, repl, text, count=1)

    def sub_literal(old, new, text, name):
        n = text.count(old)
        assert n == 1, f'[{name}] khớp {n} chỗ (cần đúng 1)'
        return text.replace(old, new)

    # 1. Không lưu epoch_N.pt (chỉ còn best.pt + epoch_latest.pt)
    s = sub_literal('if completed_epoch == args.epochs or (', 'if False or (', s, 'epoch_N')

    # 2. epoch_latest.pt: ĐẦY ĐỦ (fp32 + optimizer + scaler), ghi qua file .tmp rồi đổi tên
    pat = r'( +)if args\.save_most_recent:\n\s+torch\.save\(\n\s+checkpoint_dict,\n\s+os\.path\.join\(args\.checkpoint_path, f"epoch_latest\.pt"\),\n\s+\)[ ]*\n'
    def rep_latest(m):
        i = m.group(1)
        return (f'{i}if args.save_most_recent:  # [LITE-CKPT]\n'
                f'{i}    _tmp = os.path.join(args.checkpoint_path, "epoch_latest.tmp")\n'
                f'{i}    torch.save(checkpoint_dict, _tmp)\n'
                f'{i}    os.replace(_tmp, os.path.join(args.checkpoint_path, "epoch_latest.pt"))\n')
    s = sub_regex(pat, rep_latest, s, 'epoch_latest')

    # 3. best.pt: NHẸ (fp16, không optimizer)
    pat = r'torch\.save\(\n(\s+)checkpoint_dict,\n(\s+)os\.path\.join\(args\.checkpoint_path, f"best\.pt"\),'
    def rep_best(m):
        return ('torch.save(\n' + m.group(1) +
                '{**checkpoint_dict, "optimizer": None, "state_dict": {k: (v.half() if v.is_floating_point() else v) for k, v in checkpoint_dict["state_dict"].items()}},\n' +
                m.group(2) + 'os.path.join(args.checkpoint_path, f"best.pt"),')
    s = sub_regex(pat, rep_best, s, 'best')

    # 4. Resume được cả khi checkpoint không có optimizer (vd best.pt nhẹ)
    s = sub_literal('                    if optimizer is not None:\n                        optimizer.load_state_dict(checkpoint["optimizer"])',
                    '                    if optimizer is not None and checkpoint.get("optimizer") is not None:\n                        optimizer.load_state_dict(checkpoint["optimizer"])',
                    s, 'resume')

    # 5. Train xong đủ epoch -> tự xóa epoch_latest (giữ best.pt)
    old = '    if args.wandb and is_master(args):\n        wandb.finish()\n'
    new = ('    if is_master(args) and args.save_logs:  # [AUTO-DEL]\n'
           '        for _f in ("epoch_latest.pt", "epoch_latest.tmp"):\n'
           '            _p = os.path.join(args.checkpoint_path, _f)\n'
           '            if os.path.exists(_p):\n'
           '                os.remove(_p)\n'
           '                logging.info(f"Train xong, da xoa {_p}")\n') + old
    s = sub_literal(old, new, s, 'auto_delete')

    open(p, 'w', encoding='utf-8').write(s)
    py_compile.compile(p, doraise=True)   # báo lỗi cú pháp ngay nếu vá sai
    print('Đã vá xong:', '[LITE-CKPT]' in s, '[AUTO-DEL]' in s)
    print('best.pt nhẹ (fp16, không optimizer) | epoch_latest.pt đầy đủ | tự xóa epoch_latest khi xong')

# ---- Vá thêm: resume trên PyTorch >= 2.6 + dừng ngay nếu thiếu checkpoint ----
import py_compile
p = 'training/main.py'
s = open(p, encoding='utf-8').read().replace('\r\n', '\n')

def _sub_once(old, new, text, name):
    n = text.count(old)
    assert n == 1, f'[{name}] khớp {n} chỗ (cần đúng 1)'
    return text.replace(old, new)

if 'weights_only=False' not in s:
    s = _sub_once('torch.load(args.resume, map_location=device)',
                  'torch.load(args.resume, map_location=device, weights_only=False)',
                  s, 'weights_only')
if 'Khong tim thay checkpoint' not in s:
    s = _sub_once('            logging.info("=> no checkpoint found at \'{}\'".format(args.resume))',
                  '            raise FileNotFoundError(f"Khong tim thay checkpoint: {args.resume}")',
                  s, 'resume_missing')
open(p, 'w', encoding='utf-8').write(s)
py_compile.compile(p, doraise=True)
print('Vá resume OK:', 'weights_only=False' in s, 'Khong tim thay checkpoint' in s)

!grep -n "load-clip-weights\"" training/params.py   # phải thấy default=True



/content/PR-CLIP
2
1
2
2
False
/content/PR-CLIP
Đã vá xong: True True
best.pt nhẹ (fp16, không optimizer) | epoch_latest.pt đầy đủ | tự xóa epoch_latest khi xong
Vá resume OK: True True
450:        "--load-clip-weights", default=True, action="store_true",


## STPR reconstruction + AMP fix (applied after compatibility fixes)

In [9]:
%cd /content/PR-CLIP
import os, re, py_compile
from pathlib import Path
p = Path('model/model.py')
s = p.read_text(encoding='utf-8')
assert 'class ItraModel' in s and 'self.fusion_model = FusionModel' in s, 'Unexpected PR-CLIP model version'
if '# [STPR-v1]' not in s:
    # Real tokenizer mask for the reconstruction decoder; no change to CLIP encoding.
    old = '                text_features = open_clip_forward(texts)'
    new = '''                text_features = open_clip_forward(texts)
                # [STPR-v1] True only for actual text tokens, including EOT.
                self._stpr_text_mask = texts.ne(0).long()
                self._stpr_eot_positions = texts.argmax(dim=-1)
                if self.prompt is not None:
                    _prompt_mask = torch.ones(texts.size(0), self.n_prompt, device=texts.device, dtype=torch.long)
                    self._stpr_text_mask = torch.cat([self._stpr_text_mask[:, :1], _prompt_mask, self._stpr_text_mask[:, 1:]], dim=1)
                    self._stpr_eot_positions = self._stpr_eot_positions + self.n_prompt
'''
    assert s.count(old)==1, 'text tokenization anchor changed'
    s=s.replace(old,new,1)
    old='        text_masks = torch.ones(text_embed.size()[:-1],dtype=torch.long).to(self.device)'
    new='''        # [STPR-v1] Do not attend to padding in the opposite modality.
        text_masks = self._stpr_text_mask.to(text_embed.device)
        assert text_masks.shape == text_embed.shape[:2], (text_masks.shape, text_embed.shape)
'''
    assert s.count(old)==1, 'text mask anchor changed'
    s=s.replace(old,new,1)
    old='''        image_output = self.fusion_model(image_res_embed, image_masks, text_embed, text_masks)
        text_output = self.fusion_model(text_res_embed, text_masks, image_embed, image_masks)
'''
    new='''        # [STPR-v1] Mask 15% of visual patches and 15% of actual text tokens
        # in RECONSTRUCTION QUERIES only; CLIP encoder outputs remain intact.
        _p_img = 0.15
        _p_txt = 0.15
        _valid_img = image_masks.bool().clone()
        _valid_img[:, 0] = False  # never mask ViT CLS token
        _valid_txt = text_masks.bool().clone()
        _valid_txt[:, 0] = False  # never mask BOS token
        _valid_txt.scatter_(1, self._stpr_eot_positions[:, None], False)
        _rand_img = (torch.rand(_valid_img.shape, device=image_embed.device) < _p_img) & _valid_img
        _rand_txt = (torch.rand(_valid_txt.shape, device=text_embed.device) < _p_txt) & _valid_txt
        # Guarantee at least one token when valid tokens exist.
        for _selected, _valid in ((_rand_img, _valid_img), (_rand_txt, _valid_txt)):
            _need = (~_selected.any(dim=1)) & _valid.any(dim=1)
            if _need.any():
                _first = _valid.float().argmax(dim=1)
                _selected[_need, _first[_need]] = True
        # Zero is a dedicated masked-query representation. Original targets are untouched.
        _img_query = image_res_embed.masked_fill(_rand_img.unsqueeze(-1), 0.0)
        _txt_query = text_res_embed.masked_fill(_rand_txt.unsqueeze(-1), 0.0)
        image_output = self.fusion_model(_img_query, image_masks, text_embed, text_masks)
        text_output = self.fusion_model(_txt_query, text_masks, image_embed, image_masks)
'''
    assert s.count(old)==1, 'reconstruction anchor changed'
    s=s.replace(old,new,1)
    old='''        self.total_loss = rec_loss # + rec_loss2 # + triplet_loss + rec_loss '''
    new='''        # [STPR-v1] Local cosine reconstruction at deliberately masked positions.
        # Detached targets avoid shortcut gradients through the target branch.
        _img_token_loss = 1.0 - F.cosine_similarity(image_output.float(), image_embed.detach().float(), dim=-1)
        _txt_token_loss = 1.0 - F.cosine_similarity(text_output.float(), text_embed.detach().float(), dim=-1)
        _local_img = _img_token_loss[_rand_img].mean()
        _local_txt = _txt_token_loss[_rand_txt].mean()
        _local_loss = (_local_img + _local_txt) * 0.5
        self.total_loss = rec_loss + 0.05 * _local_loss  # beta=0.05, first experiment
'''
    assert s.count(old)==1, 'reconstruction loss anchor changed'
    s=s.replace(old,new,1)
    p.write_text(s,encoding='utf-8')
# Backward/optimizer fix: accumulate 4 batches in BOTH AMP and fp32.
p=Path('training/train.py'); s=p.read_text(encoding='utf-8')
if '# [STPR-AMP-FIX]' not in s:
    marker='    step_lou = 4\n'
    assert s.count(marker)==1, 'accumulation anchor changed'
    s=s.replace(marker, marker+'    optimizer.zero_grad(set_to_none=True)  # [STPR-AMP-FIX]\n    norm = torch.tensor(0.0)\n',1)
    start='''            if scaler is not None:
                scaler.scale(total_loss).backward()'''
    end='''            batch_time_m.update(time.time() - end)'''
    assert s.count(start)==1 and s.count(end)==1,'AMP anchors changed'
    a=s.index(start); b=s.index(end,a)
    replacement='''            # [STPR-AMP-FIX] Unscale and step ONLY at accumulation boundary.
            if scaler is not None:
                scaler.scale(total_loss).backward()
            else:
                total_loss.backward()
            _step_now = ((i + 1) % step_lou == 0 or i == train_loader_size - 1)
            if _step_now:
                if scaler is not None:
                    scaler.unscale_(optimizer)
                norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=args.max_grad_norm)
                if scaler is not None:
                    if args.horovod:
                        optimizer.synchronize()
                        with optimizer.skip_synchronize():
                            scaler.step(optimizer)
                    else:
                        scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)

'''
    s=s[:a]+replacement+s[b:]
    s=s.replace('f"LR: {optimizer.param_groups[0][\'lr\']:3f} "','f"LR: {optimizer.param_groups[0][\'lr\']:.3e} "')
    p.write_text(s,encoding='utf-8')
for f in ('model/model.py','training/train.py'):
    py_compile.compile(f,doraise=True)
print('STPR-v1 + AMP accumulation patch: PASSED compile')


/content/PR-CLIP
STPR-v1 + AMP accumulation patch: PASSED compile


In [10]:
# Verify actual code before starting an expensive run
from pathlib import Path
import py_compile
for p in ['model/model.py', 'training/train.py']:
    py_compile.compile(p, doraise=True)
assert '[STPR-v1]' in Path('model/model.py').read_text()
assert '[STPR-AMP-FIX]' in Path('training/train.py').read_text()
assert Path('cache/weights/RET-2.pt').exists(), 'Missing RET-2 pretrained weights'
print('PASS: architecture patch, AMP patch, pretrained checkpoint present')

PASS: architecture patch, AMP patch, pretrained checkpoint present


## First training experiment: 1 epoch, batch 16, LR 2e-7

Output name is intentionally unique. Compare with your existing PR-CLIP baseline and RET-2 without fine-tuning. The first run uses the same dataset and main hyperparameters as your earlier best run, but the AMP fix changes optimizer dynamics, so direct interpretation requires a baseline rerun with the same fix.

In [11]:
%cd /content/PR-CLIP
!torchrun --nproc_per_node 1 -m training.main \
  --train-data ./dataset/train/ret2.csv --images-dir ./images \
  --retrieval-data ./dataset/test/rsicd_test.csv --retrieval-images-dir ./images \
  --csv-separator , --csv-img-key filename \
  --retrieval-csv-separator , --retrieval-csv-img-key filename \
  --retrieval-frequency 1 --save-frequency 0 \
  --epochs 1 --batch-size 16 --workers 2 --precision amp \
  --lr 2e-7 --warmup 100 --weight_decay 0.5 --max-grad-norm 50 \
  --image-model ViT-B-32 --image-model-builder openclip \
  --text-model ViT-B-32 --text-model-builder openclip \
  --pretrained-image-model --pretrained-text-model --loss InfoNCE \
  --seed 1 \
  --name rsicd_STPR_mask15_beta005_bs16_lr2e-7 --logs /content/drive/MyDrive/prclip_logs

/content/PR-CLIP
I0000 00:00:1793712426.333600    1924 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793712428.709935    1924 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/PR-CLIP/evaluation/SentEval/senteval/utils.py:62: SyntaxWarning: invalid escape sequence '\d'
  assert re.match("^[+-]?(\d+(\.\d*)?|\.\d+)$", split[1]) is not None
Namespace(config_yaml=None, text_model_builder='openclip', text_model='ViT-B-32', text_pooler='cls', text_mo